# SkinCancerAI — Training auf Google Colab (kostenlose GPU)

> ⚠️ **Kein Medizinprodukt.** Nur für Forschungs-/Lernzwecke. Ersetzt keine ärztliche Diagnose.

**Vor dem Start:** `Laufzeit → Laufzeittyp ändern → GPU (T4)` wählen.

Diese Pipeline: klinische ISIC-Bilder (Fitzpatrick I–III) → EfficientNet-Fine-Tuning → TFLite-Export.

In [ ]:
# 1) Code holen und Abhängigkeiten installieren
# Variante A: eigenes Repo klonen
# !git clone <DEIN_REPO_URL>
# Variante B: Dateien manuell hochladen (linke Seitenleiste -> Dateien)
# WICHTIG: Pfad an deinen Ordnernamen anpassen (z. B. /content/dermascreen)
%cd /content/dermascreen
!pip install -q -r requirements.txt
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'  # Keras-2-Verhalten (noetig ab TF 2.16 auf Colab)
# Falls Colab jetzt 'Sitzung neu starten' anbietet -> anklicken und ab dieser Zelle weiter.

In [ ]:
# 2) GPU prüfen
import tensorflow as tf
print('TF:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

In [ ]:
# 3) Daten: ISIC-Metadaten laden, filtern (Fitzpatrick I-III + klinisch), splitten
!python data/download_isic.py   --config config.yaml
!python data/filter_metadata.py --config config.yaml
!python data/prepare_dataset.py --config config.yaml

In [ ]:
# 4) Training (zwei Phasen: Kopf -> Fine-Tuning)
!python src/train.py --config config.yaml

In [ ]:
# 5) Klinische Evaluierung auf dem strikt getrennten Test-Set
!python src/evaluate.py --config config.yaml

In [ ]:
# 6) Export nach TFLite für die Android-App
!python src/export_tflite.py --config config.yaml
from google.colab import files
files.download('models/skincancer.tflite')
files.download('models/model_meta.json')